# 基于FFT频域特征与CNN的车牌字符识别 —— 可视化分析

本Notebook展示完整的信号处理流程：
1. 图像预处理（灰度化、去噪）
2. FFT频域变换与频谱分析
3. 高通/低通滤波器效果对比
4. 不同噪声水平下的频域特征变化
5. 训练结果可视化（由同学B的代码生成数据后使用）

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial']
matplotlib.rcParams['axes.unicode_minus'] = False

from src.preprocess import (
    to_grayscale, denoise_gaussian, denoise_median,
    binarize, add_gaussian_noise, add_salt_pepper_noise,
    preprocess_pipeline
)
from src.fft_features import (
    fft2d, ifft2d, extract_fft_features,
    apply_filter, gaussian_highpass_filter, gaussian_lowpass_filter,
    plot_fft_pipeline, plot_filter_comparison,
    plot_noise_fft_comparison, plot_magnitude_and_phase
)
from config import CHAR_DIR, RESULTS_DIR

os.makedirs(RESULTS_DIR, exist_ok=True)
print('模块加载完成！')

## 1. 加载示例字符图像

如果还没有运行 `prepare_data.py` 处理CCPD数据集，这里用程序生成一些模拟字符图片做演示。

In [ ]:
def load_sample_images(n=5):
    """从已处理的字符数据集加载样本，若不存在则生成模拟字符。"""
    samples = []
    labels = []
    
    if os.path.exists(CHAR_DIR):
        for label_dir in sorted(os.listdir(CHAR_DIR))[:n]:
            label_path = os.path.join(CHAR_DIR, label_dir)
            if not os.path.isdir(label_path):
                continue
            # 查找图片（可能在train子目录或直接在目录下）
            for sub in ['train', '.']:
                search_dir = os.path.join(label_path, sub) if sub != '.' else label_path
                if not os.path.exists(search_dir):
                    continue
                imgs = [f for f in os.listdir(search_dir) if f.endswith(('.jpg', '.png'))]
                if imgs:
                    img = cv2.imread(os.path.join(search_dir, imgs[0]), cv2.IMREAD_GRAYSCALE)
                    if img is not None:
                        samples.append(img)
                        labels.append(label_dir)
                        break
            if len(samples) >= n:
                break
    
    # 如果没有真实数据，生成模拟字符
    if len(samples) == 0:
        print('未找到真实数据，生成模拟字符图像...')
        fake_chars = ['A', 'B', '3', 'K', '7']
        for ch in fake_chars:
            img = np.zeros((20, 20), dtype=np.uint8)
            cv2.putText(img, ch, (2, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.6, 255, 1)
            samples.append(img)
            labels.append(ch)
    
    return samples, labels

samples, labels = load_sample_images()

fig, axes = plt.subplots(1, len(samples), figsize=(3*len(samples), 3))
for ax, img, lbl in zip(axes, samples, labels):
    ax.imshow(img, cmap='gray')
    ax.set_title(lbl, fontsize=14)
    ax.axis('off')
plt.suptitle('示例字符图像', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'sample_chars.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2. 图像预处理效果展示

展示灰度化 → 去噪 → 二值化的完整流程。

In [ ]:
# 取第一个样本做演示
demo_img = samples[0]

# 放大到更容易观察的尺寸
demo_big = cv2.resize(demo_img, (80, 80), interpolation=cv2.INTER_NEAREST)

# 各种预处理
noisy = add_gaussian_noise(demo_big, sigma=30)
denoised_gauss = denoise_gaussian(noisy, ksize=3)
denoised_median = denoise_median(noisy, ksize=3)
binary = binarize(denoised_gauss)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
titles = ['原始图像', '加噪(σ=30)', '高斯去噪', '中值去噪', 'Otsu二值化']
images = [demo_big, noisy, denoised_gauss, denoised_median, binary]

for ax, title, img in zip(axes, titles, images):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=13)
    ax.axis('off')

plt.suptitle('图像预处理流程', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'preprocess_pipeline.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. FFT频域分析 —— 完整流程

**核心流程：** 原图 → FFT幅度谱 → 高通滤波器 → 滤波后频谱 → 逆FFT恢复

这是论文的重点图！

In [ ]:
demo_big = cv2.resize(samples[0], (80, 80), interpolation=cv2.INTER_NEAREST)

plot_fft_pipeline(demo_big, sigma=10, 
                  save_path=os.path.join(RESULTS_DIR, 'fft_pipeline.png'))

## 4. 幅度谱与相位谱

In [ ]:
plot_magnitude_and_phase(demo_big, 
                         save_path=os.path.join(RESULTS_DIR, 'magnitude_phase.png'))

## 5. 不同滤波器效果对比

对比高斯高通、理想高通、巴特沃斯高通、高斯低通的效果差异。

In [ ]:
plot_filter_comparison(demo_big, 
                       save_path=os.path.join(RESULTS_DIR, 'filter_comparison.png'))

## 6. 不同σ参数下高通滤波效果

σ越小 → 滤波越强（去除更多低频） → 只保留最强的边缘

In [ ]:
sigmas = [3, 5, 10, 20, 50]

fig, axes = plt.subplots(2, len(sigmas), figsize=(4*len(sigmas), 7))

for i, sigma in enumerate(sigmas):
    # 滤波器本身
    H = gaussian_highpass_filter(demo_big.shape, sigma=sigma)
    # 滤波后图像
    filtered, _, _, _ = apply_filter(demo_big, 'gaussian_high', sigma=sigma)
    
    axes[0, i].imshow(H, cmap='gray')
    axes[0, i].set_title(f'滤波器 σ={sigma}', fontsize=12)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(filtered, cmap='gray')
    axes[1, i].set_title(f'滤波结果', fontsize=12)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('滤波器', fontsize=13)
axes[1, 0].set_ylabel('滤波结果', fontsize=13)
plt.suptitle('高斯高通滤波器 σ 参数影响', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'sigma_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. 不同噪声水平下的FFT对比

展示噪声如何影响频谱，以及高通滤波的抗噪效果。

In [ ]:
noise_levels = [0, 10, 20, 30, 50]

fig, axes = plt.subplots(3, len(noise_levels), figsize=(4*len(noise_levels), 10))

for i, sigma_noise in enumerate(noise_levels):
    if sigma_noise == 0:
        noisy = demo_big.copy()
    else:
        noisy = add_gaussian_noise(demo_big, sigma=sigma_noise)
    
    _, mag, _ = fft2d(noisy)
    filtered = extract_fft_features(noisy, sigma=10)
    
    axes[0, i].imshow(noisy, cmap='gray')
    axes[0, i].set_title(f'噪声 σ={sigma_noise}', fontsize=12)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(mag, cmap='hot')
    axes[1, i].set_title('FFT频谱', fontsize=12)
    axes[1, i].axis('off')
    
    axes[2, i].imshow(filtered, cmap='gray')
    axes[2, i].set_title('高通滤波后', fontsize=12)
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('含噪图像', fontsize=13)
axes[1, 0].set_ylabel('频谱', fontsize=13)
axes[2, 0].set_ylabel('滤波结果', fontsize=13)
plt.suptitle('不同噪声水平下的FFT频域分析', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'noise_fft_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. 多个字符的FFT特征对比

观察不同字符的频域特征差异。

In [ ]:
n = len(samples)
fig, axes = plt.subplots(3, n, figsize=(4*n, 10))

for i, (img, lbl) in enumerate(zip(samples, labels)):
    img_big = cv2.resize(img, (80, 80), interpolation=cv2.INTER_NEAREST)
    _, mag, _ = fft2d(img_big)
    feat = extract_fft_features(img_big, sigma=10)
    
    axes[0, i].imshow(img_big, cmap='gray')
    axes[0, i].set_title(lbl, fontsize=14)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(mag, cmap='hot')
    axes[1, i].set_title('频谱', fontsize=12)
    axes[1, i].axis('off')
    
    axes[2, i].imshow(feat, cmap='gray')
    axes[2, i].set_title('高通特征', fontsize=12)
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('原图', fontsize=13)
axes[1, 0].set_ylabel('FFT频谱', fontsize=13)
axes[2, 0].set_ylabel('特征图', fontsize=13)
plt.suptitle('不同字符的FFT频域特征对比', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'multi_char_fft.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. 滤波器频率响应3D图

论文中展示滤波器形状的3D可视化。

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

shape = (80, 80)
H_hp = gaussian_highpass_filter(shape, sigma=10)
H_lp = gaussian_lowpass_filter(shape, sigma=10)

x = np.arange(shape[1])
y = np.arange(shape[0])
X, Y = np.meshgrid(x, y)

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X, Y, H_hp, cmap='viridis', alpha=0.8)
ax1.set_title('高斯高通滤波器 (σ=10)', fontsize=13)
ax1.set_xlabel('u')
ax1.set_ylabel('v')
ax1.set_zlabel('H(u,v)')

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(X, Y, H_lp, cmap='viridis', alpha=0.8)
ax2.set_title('高斯低通滤波器 (σ=10)', fontsize=13)
ax2.set_xlabel('u')
ax2.set_ylabel('v')
ax2.set_zlabel('H(u,v)')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'filter_3d.png'), dpi=150, bbox_inches='tight')
plt.show()

---

## 以下部分需要同学B的训练结果数据

运行完 `src/train.py` 和 `scripts/run_experiments.py` 后，会在 `results/` 目录生成训练日志，
以下代码读取这些数据绘制训练曲线和实验对比图。

In [ ]:
import json

def load_training_log(path):
    """加载训练日志 JSON 文件。"""
    if not os.path.exists(path):
        print(f'训练日志不存在: {path}')
        print('请先运行 src/train.py 生成训练数据')
        return None
    with open(path, 'r') as f:
        return json.load(f)

# 尝试加载
baseline_log = load_training_log(os.path.join(RESULTS_DIR, 'train_log_baseline.json'))
fft_log = load_training_log(os.path.join(RESULTS_DIR, 'train_log_fft.json'))

In [ ]:
# 训练曲线对比（Baseline vs FFT+CNN）
if baseline_log and fft_log:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs_b = range(1, len(baseline_log['train_loss']) + 1)
    epochs_f = range(1, len(fft_log['train_loss']) + 1)
    
    ax1.plot(epochs_b, baseline_log['train_loss'], 'b-', label='Baseline 训练')
    ax1.plot(epochs_b, baseline_log['val_loss'], 'b--', label='Baseline 验证')
    ax1.plot(epochs_f, fft_log['train_loss'], 'r-', label='FFT+CNN 训练')
    ax1.plot(epochs_f, fft_log['val_loss'], 'r--', label='FFT+CNN 验证')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('训练损失曲线对比')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs_b, baseline_log['val_acc'], 'b-o', label='Baseline')
    ax2.plot(epochs_f, fft_log['val_acc'], 'r-o', label='FFT+CNN')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('验证准确率曲线对比')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('等待训练完成后再运行此cell')

In [ ]:
# 噪声鲁棒性对比柱状图
noise_results_path = os.path.join(RESULTS_DIR, 'noise_experiment.json')
if os.path.exists(noise_results_path):
    with open(noise_results_path, 'r') as f:
        noise_results = json.load(f)
    
    noise_levels = noise_results['noise_levels']
    baseline_acc = noise_results['baseline_acc']
    fft_acc = noise_results['fft_acc']
    
    x = np.arange(len(noise_levels))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars1 = ax.bar(x - width/2, baseline_acc, width, label='Baseline CNN', color='#4472C4')
    bars2 = ax.bar(x + width/2, fft_acc, width, label='FFT + CNN', color='#ED7D31')
    
    ax.set_xlabel('噪声标准差 σ', fontsize=13)
    ax.set_ylabel('识别准确率 (%)', fontsize=13)
    ax.set_title('不同噪声水平下的识别准确率对比', fontsize=15)
    ax.set_xticks(x)
    ax.set_xticklabels([str(n) for n in noise_levels])
    ax.legend(fontsize=12)
    ax.grid(True, axis='y', alpha=0.3)
    
    # 在柱状图上标数值
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=10)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'noise_robustness.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('等待噪声实验完成后再运行此cell')

## 总结

从以上可视化可以看出：
1. **FFT频谱**能清晰展示字符的频域特征，不同字符具有不同的频谱分布
2. **高通滤波**能有效增强字符边缘信息，抑制背景干扰
3. 在噪声条件下，**频域滤波+CNN**的识别鲁棒性优于单纯CNN
4. 高斯高通滤波器的**σ参数**对滤波强度有显著影响，需要根据实际情况调优